In [ ]:
import Pkg; Pkg.add(["SpecialFunctions"])


In [ ]:
using Random, Distributions, SpecialFunctions


# Parameters

This section sets the model's fundamentals and structural parameters, and fixes the indexing convention used throughout the rest of the notebook.

**Regions and markets.** There are $N$ regions and $J$ real sectors, plus a non-employment "sector 0" in every region, giving $M = J+1$ markets per region. Any array indexed over every possible market a household could occupy (labor $L$, value function $V$, migration shares $\mu$, mobility costs $\tau$) is `N × M`, with **column 1 = non-employment** and **columns 2:M = real sectors 1..J**. Arrays that only pertain to production ($A$, $w$, $\kappa$, $\theta$, $\eta$, $\gamma$, $\alpha$) are `N × J`, since sector 0 has no production, wage, or price — to compare a market-indexed array against a sector-indexed one, offset the sector index by $+1$.

**Time-varying fundamentals**, $\Theta_t = (A_t, \kappa_t)$:
- $A_t^{nj}$ — productivity in region $n$, sector $j$
- $\kappa_t^{nj,ij}$ — iceberg trade cost shipping sector-$j$ goods from region $i$ to region $n$

**Constant fundamentals**, $\bar\Theta = (\Upsilon, b)$:
- $\Upsilon = \{\tau^{nj,ik}\}$ — labor relocation (mobility) costs, in utility terms, from market $(n,j)$ to market $(i,k)$
- $b^n$ — value of home production in region $n$ (a non-employed household's consumption)

**Structural parameters:**
- $\beta \in [0,1)$ — discount factor
- $\theta^j$ — Fréchet trade elasticity in sector $j$
- $\nu$ — dispersion of the idiosyncratic migration taste shock ($1/\nu$ is the migration elasticity)
- $\alpha^j$ — Cobb-Douglas consumption share on sector $j$, with $\sum_j \alpha^j = 1$

**State variable:** $L_t = \{L_t^{nj}\}$, the mass of households in each market at time $t$ — the only object carrying information from one period to the next.

In [ ]:
#try indexing format of J[region]_[sector]_[time], x if not indexed by that component

N = 2 # Number of regions
J = 3 # Number of real sectors (sector 0 / non-employment is handled separately -- see M below)
M = J + 1 # Number of markets per region: non-employment (market column 1, i.e. sector 0) plus
          # the J real sectors (market columns 2:M, i.e. sectors 1:J). Per the model, a "market"
          # is a region-sector pair (n,j) with j = 0,...,J, so any array indexed over "every
          # market a household could be in" (L, V, mu, tau_mig) has rows = regions (N), columns
          # = markets (M), with column 1 = non-employment. Arrays that only pertain to real
          # production (A, w, kappa, theta, eta, gamma, alpha) keep rows = regions, columns =
          # real sectors (J) -- to look one of these up against a market-indexed array, offset
          # the sector index by +1 (real sector j lives in market column j+1).
n_omega = 10000 # number of varieties used to discretize the omega in [0,1] continuum per region-sector


L_0  = ones(N,M) #labor force in economy at time 0, over all N regions x M markets (incl. non-employment)

Random.seed!(1) # Seed for the random number generator (to guarantee reproducibility; this is standard in research these days)

A_0 = rand(N,J) #region-sector productivity (real sectors only; undefined for non-employment)

B = ones(N,J) #arbitrary coefficient

w_0 = ones(N,J) # wage, real sectors only -- non-employment pays no wage; households there consume b_n instead

b = ones(N) # value of home production: consumption of a non-employed household in region n

kappa_0 = [ones(N,N) for _ in 1:J] # iceberg trade costs, per sector (kappa: trade cost)

# sigma = 2 # Substitution elasticity between goods
# L = ones(N, 1) # Size of labor force in each country

theta = fill(4.0, J) # Frechet shape parameter per sector (governs dispersion of productivity draws
                      # across the continuum of varieties/regions, and the trade elasticity);
                      # scale is absorbed into A_0, so no separate T parameter is needed

eta = fill(2.0, N, J) # elasticity of substitution across varieties within sector j (CES aggregator)

gamma = ones(N, J) # labor/value-added share of production; = 1 since this simplified model has no materials

beta = 0.95 # household discount factor
# utility cost of moving from market (n,j) to market (i,k), j,k = 0,...,J (market columns 1:M,
# where column 1 is non-employment); 0 to stay in the same market, 1 otherwise (tau: migration cost)
tau_mig = [(n == i && j == k) ? 0.0 : 1.0 for n in 1:N, j in 1:M, i in 1:N, k in 1:M]

alpha = rand(J) # Cobb-Douglas consumption shares: alpha[g] is the share of household income
alpha = alpha ./ sum(alpha) # spent on good g, common across all regions/sectors; normalized to sum to 1

nu = 1.0 # dispersion of the idiosyncratic (Frechet/type-I EV) taste shock over migration destinations
         # (larger nu = migration is less sensitive to utility differences, i.e. more friction)

T = 10 # number of periods to simulate the labor dynamics forward


# Two-Period Household Block (Wages Still Exogenous)

This section extends the static, single-period Migration Decision block from Notebook 1
(`01_static_migration_decision.ipynb`) to two periods, $t=0,1$. Wages are still given by hand,
not solved for by market clearing (that's introduced in Notebook 3) -- but unlike Notebook 1,
where a single wage $w_0$ was assumed to hold forever (so $V$ had a single stationary fixed
point), here $w_0$ and $w_1$ are allowed to differ. This is the minimal step needed to motivate
**backward induction**: with $w_0 \ne w_1$, $V_0$ is no longer the same object as the stationary
fixed point -- it depends on $V_1$, which in turn is pinned down by treating period 1 as if its
conditions held forever after (the same terminal-condition device used in the Sequential
Equilibrium section of Notebook 3, restricted here to a 2-period horizon).

**Flow utility and value functions.** `flow_utility_mkt(w)` and `stationary_V(U_mkt)` package up
the static production/household block (Notebook 1's Production-Intermediates and Household
Problem sections) and the Bellman fixed point (Notebook 1's Migration Decision section) as
reusable functions of an arbitrary wage matrix `w`, since flow utility must now be evaluated at
two different wage levels instead of one fixed global.

In [ ]:
# flow_utility_mkt(w): production + household consumption block (Notebook 1's Production-
# Intermediates + Household Problem sections), packaged as a function of an arbitrary wage
# matrix w instead of the fixed global w_0, so it can be evaluated at both w_0 and w_1 below.
function flow_utility_mkt(w::AbstractMatrix)
    x = B .* w
    trade_cost_term = [
        (x[i,j] * kappa_0[j][n,i])^(-theta[j]) * A_0[i,j]^(theta[j]*gamma[i,j])
        for n in 1:N, j in 1:J, i in 1:N
    ]
    Gamma = [SpecialFunctions.gamma((theta[j] + 1 - eta[n,j]) / theta[j])^(1 / (1 - eta[n,j])) for n in 1:N, j in 1:J]
    P = [Gamma[n,j] * sum(trade_cost_term[n,j,:])^(-1/theta[j]) for n in 1:N, j in 1:J]
    P_hat = [prod((P[n,g] / alpha[g])^alpha[g] for g in 1:J) for n in 1:N]
    C = [w[n,j] / P_hat[n] for n in 1:N, j in 1:J] # real consumption of an employed household
    return hcat(b, C) # N x M, column 1 = non-employment (fixed consumption b^n)
end

# stationary_V(U_mkt): the Bellman fixed point from Notebook 1's Migration Decision section
# (value function iteration, a contraction since beta<1), packaged as a function of flow utility
# U_mkt instead of the fixed global U_0_mkt. Used below as the terminal condition for period 1:
# "period 1's conditions hold forever after."
function stationary_V(U_mkt::AbstractMatrix; tol=1e-12, maxiter=10_000)
    V = log.(U_mkt)
    for _ in 1:maxiter
        V_next = [
            log(U_mkt[n,j]) + nu * log(sum(exp((beta*V[i,k] - tau_mig[n,j,i,k]) / nu) for i in 1:N, k in 1:M))
            for n in 1:N, j in 1:M
        ]
        if maximum(abs.(V_next .- V)) < tol
            V = V_next
            break
        end
        V = V_next
    end
    return V
end

# migration_shares(V_next): the logit migration shares (eq 3), as a function of next period's
# value function -- same closed form as Notebook 1's mu_0, but now V_next need not be a
# stationary fixed point.
function migration_shares(V_next::AbstractMatrix)
    [
        exp((beta*V_next[i,k] - tau_mig[n,j,i,k]) / nu) /
        sum(exp((beta*V_next[m,h] - tau_mig[n,j,m,h]) / nu) for m in 1:N, h in 1:M)
        for n in 1:N, j in 1:M, i in 1:N, k in 1:M
    ]
end


**Exogenous wage path.** `w_0` (from Parameters) is period 0's wage, unchanged from Notebook 1.
Period 1's wage `w_1` is a second, hand-specified exogenous level -- a 20% wage bump in region 1,
sector 1, with everything else unchanged -- chosen only so that $w_0 \ne w_1$ and the two-period
backward induction actually does something different from Notebook 1's single stationary fixed
point. (This mirrors the shape of the shock experiments in Notebook 4's Sandbox section, but here
the wage change is simply asserted, not solved for by market clearing.)

In [ ]:
w_1 = copy(w_0)
w_1[1,1] *= 1.2 # exogenous +20% wage bump, region 1 sector 1, at t=1


**Backward induction over $t=0,1$.** $V_1$ is the terminal condition: the stationary fixed point
under $w_1$, standing in for "period 1's conditions hold forever after" beyond this 2-period
horizon. $V_0$ is then a *single* direct evaluation of the Bellman equation using the already-known
$V_1$ and period 0's flow utility -- no fixed-point iteration needed, since only the terminal value
function requires one. Migration shares $\mu_0$ come from $V_1$ (eq 3), and the labor distribution
$L_1$ is simulated one step forward from $L_0$ via the law of motion (eq 4), exactly as in
Notebook 1's Labor Dynamics section.

In [ ]:
U_mkt_0 = flow_utility_mkt(w_0)
U_mkt_1 = flow_utility_mkt(w_1)

V_1 = stationary_V(U_mkt_1) # terminal condition: period 1's conditions hold forever after

V_0 = [
    log(U_mkt_0[n,j]) + nu * log(sum(exp((beta*V_1[i,k] - tau_mig[n,j,i,k]) / nu) for i in 1:N, k in 1:M))
    for n in 1:N, j in 1:M
] # single backward-induction step from V_1 -- no fixed-point iteration needed here

mu_0 = migration_shares(V_1) # eq 3, using next period's (terminal) value function

L_1 = [
    sum(mu_0[i,k,n,j] * L_0[i,k] for i in 1:N, k in 1:M)
    for n in 1:N, j in 1:M
] # eq 4, one step forward


In [ ]:
function print_labeled(title, description, M::AbstractMatrix, colnames)
    println(title)
    println("  ", description)
    println("             " * join(rpad.(colnames, 16)))
    for n in 1:size(M,1)
        println("  Region $n:  " * join([rpad(string(round(M[n,c], digits=4)), 16) for c in 1:size(M,2)]))
    end
    println()
end


In [ ]:
println("="^70)
println("PERIOD t = 0")
println("="^70)
print_labeled("Labor distribution L[0]", "mass of households currently in each region-market",
    L_0, ["Non-employment","Sector 1","Sector 2","Sector 3"])
print_labeled("Exogenous wage w[0]", "hand-specified, not market-clearing",
    w_0, ["Sector 1","Sector 2","Sector 3"])
print_labeled("Value function V[0]", "expected discounted lifetime utility of a household in each region-market",
    V_0, ["Non-employment","Sector 1","Sector 2","Sector 3"])

println("="^70)
println("PERIOD t = 1")
println("="^70)
print_labeled("Labor distribution L[1]", "mass of households currently in each region-market",
    L_1, ["Non-employment","Sector 1","Sector 2","Sector 3"])
print_labeled("Exogenous wage w[1]", "hand-specified, not market-clearing",
    w_1, ["Sector 1","Sector 2","Sector 3"])
print_labeled("Value function V[1]", "expected discounted lifetime utility of a household in each region-market",
    V_1, ["Non-employment","Sector 1","Sector 2","Sector 3"])
